<a href="https://colab.research.google.com/github/naheejeong/cs182-final-project/blob/main/02_llama_full_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# hardware check - GPU & memory assigned
!nvidia-smi

Sun Nov 22 03:08:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Connect Google Drive to the Colab machine so results are saved to Drive.
from google.colab import drive
drive.mount('/content/drive')

# Setup commands (! = system command, % = notebook command)
%cd /content

# Download the InjecAgent benchmark (attack cases and evaluation code) from GitHub.
# --> this makes a new folder named "InjecAgent" in /content.
!git clone https://github.com/uiuc-kang-lab/InjecAgent.git
%cd /content/InjecAgent

# Install the 6 required packages (4 from requirements.txt + together and nltk).
!pip install -q -r requirements.txt together nltk

# Create a results folder in Google Drive (My Drive/cs182/results).
!mkdir -p /content/drive/MyDrive/cs182/results

# Create shortcut: InjecAgent/results -> My Drive/cs182/results, so outputs are saved to Drive.
!ln -sfn /content/drive/MyDrive/cs182/results results

# Provide environment variables needed by the InjecAgent code.
import os
os.environ["PYTHONPATH"] = "/content/InjecAgent"
os.environ["OPENAI_API_KEY"] = "unused" # placeholder; we never use!

!ls

Mounted at /content/drive
/content
Cloning into 'InjecAgent'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 102 (delta 46), reused 35 (delta 35), pack-reused 20 (from 1)
Receiving objects: 100% (102/102), 1.56 MiB | 4.97 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/InjecAgent
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
asset  data  LICENCE  README.md  requirements.txt  results  src


In [3]:
# Log in to HF using the token stored in Secrets. Token name: HF_TOKEN, username: naheejeong
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()['name'])

# Testing Llama access - download one tiny file from the model's page.
config_path = hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json")
print(config_path)

Logged in as: naheejeong


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/config.json


In [4]:
# Our wrapper class that lets InjecAgent run Llama 3.1 (developed and tested in 01_llama_vulnerability.ipynb).
# It is kept as text here and written into InjecAgent's src/models.py, where InjecAgent's script looks for it.
marker = "# ==== added by 02_llama_full_run.ipynb ===="

llama3_code = marker + r'''

class Llama3Model(BaseModel):
    # Response length limit: 512 new tokens, the limit InjecAgent uses for open-source models.
    # InjecAgent's evaluator reads at most the first 500 words, which 512 tokens does not reach in normal text.
    MAX_NEW_TOKENS = 512

    def __init__(self, params):
        super().__init__()
        from torch import bfloat16
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tok = AutoTokenizer.from_pretrained(params['model_name'])
        self.model = AutoModelForCausalLM.from_pretrained(
            params['model_name'], dtype=bfloat16, device_map="auto")    # half precision, placed on the GPU automatically

    def prepare_input(self, sys_prompt, user_prompt_filled):
        return [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt_filled},
        ]

    def call_model(self, model_input):    # Whatever prepare_input returns is the model input
        from torch import no_grad

        # 1. Format the messages AND turn them into token IDs
        inputs = self.tok.apply_chat_template(
            model_input,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device)

        # 2. Let Llama generate the continuation
        with no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=self.tok.eos_token_id,
            )

        # 3. Keep only the new tokens (the reply) and turn them back into text
        all_tokens = out[0]
        prompt_length = inputs["input_ids"].shape[-1]
        reply_tokens = all_tokens[prompt_length:]
        return self.tok.decode(reply_tokens, skip_special_tokens=True)

MODELS["Llama3"] = Llama3Model    # register the class under the name used by --model_type
'''

# Write the class to the end of InjecAgent's models.py, replacing any earlier copy of ours.
models_path = "/content/InjecAgent/src/models.py"
original = open(models_path).read().split(marker)[0].rstrip() + "\n\n"    # InjecAgent's own code, without our part
with open(models_path, "w") as f:
    f.write(original + llama3_code)

# Confirm InjecAgent now knows the class. (This does not load the model.)
import importlib, src.models
importlib.reload(src.models)
print("registered model types:", list(src.models.MODELS))

registered model types: ['Claude', 'GPT', 'Llama', 'TogetherAI', 'Llama3']


In [5]:
# Record the exact setup next to the results, for the write-up. One line is added per session.
import json, subprocess, datetime, torch, transformers

def run(cmd):    # run a system command and return what it prints
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

run_info = {
    "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "gpu": run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "injecagent_commit": run(["git", "-C", "/content/InjecAgent", "rev-parse", "HEAD"]),
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "model_revision": config_path.split("/snapshots/")[1].split("/")[0],
    "prompt_type": "InjecAgent", "max_new_tokens": 512, "sampling": False, "only_first_step": True,
}
with open("/content/drive/MyDrive/cs182/results/run_info.jsonl", "a") as f:
    f.write(json.dumps(run_info) + "\n")
print(json.dumps(run_info, indent=1))

{
 "recorded_at_utc": "2026-11-22T03:09:04.347987+00:00",
 "gpu": "NVIDIA L4, 23034 MiB",
 "torch": "2.14.1+cu130",
 "transformers": "5.22.1",
 "injecagent_commit": "f19c9f2c79a41046eb13c03c51a24c567a8ffa07",
 "model": "meta-llama/Llama-3.1-8B-Instruct",
 "model_revision": "0e9e39f249a16976918f6564b8830bc894c89659",
 "prompt_type": "InjecAgent",
 "max_new_tokens": 512,
 "sampling": false,
 "only_first_step": true
}


In [6]:
# Back up the results folder into a time-stamped copy before running or resuming.
import shutil
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = f"/content/drive/MyDrive/cs182/results_backup/{stamp}"
shutil.copytree("/content/drive/MyDrive/cs182/results", backup_dir)
print("backed up to", backup_dir)

backed up to /content/drive/MyDrive/cs182/results_backup/20261122_030950


In [ ]:
# Run (or resume) the base setting. --use_cache reuses every case already saved.
%cd /content/InjecAgent
!python3 src/evaluate_prompted_agent.py --model_type Llama3 --model_name meta-llama/Llama-3.1-8B-Instruct --setting base --prompt_type InjecAgent --only_first_step --use_cache

/content/InjecAgent
{'model_type': 'Llama3', 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'setting': 'base', 'prompt_type': 'InjecAgent', 'only_first_step': True, 'use_cache': True, 'only_get_score': False}
tokenizer_config.json: 100% 55.4k/55.4k [00:00<00:00, 92.1MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 30.9MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.77MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 2.80MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  19% 3.06G/16.1G [00:13<01:14, 174MB/s,  336MB/s  ]

Fetching 4 files:  25% 1/4 [00:14<00:42, 14.14s/it]
Reconstructing (incomplete total...):  72% 11.6G

In [ ]:
# Run (or resume) the enhanced setting.
%cd /content/InjecAgent
!python3 src/evaluate_prompted_agent.py --model_type Llama3 --model_name meta-llama/Llama-3.1-8B-Instruct --setting enhanced --prompt_type InjecAgent --only_first_step --use_cache

In [ ]:
# Check that every case was saved, tally the labels, and print the scores for each setting.
import os, json
from collections import Counter
from src.utils import get_score

res_dir = "/content/InjecAgent/results/prompted_Llama3_meta-llama/Llama-3.1-8B-Instruct_InjecAgent"
expected = {"dh": 510, "ds": 544}    # dh = direct harm, ds = data stealing

for setting in ["base", "enhanced"]:
    files = {a: f"{res_dir}/test_cases_{a}_{setting}.json" for a in ["dh", "ds"]}
    if not all(os.path.exists(p) for p in files.values()):
        print(setting, ": not finished yet\n")
        continue

    # 1. Completeness: how many cases were saved in each file, and their labels.
    all_items = []
    for a, p in files.items():
        items = [json.loads(line) for line in open(p)]
        all_items += items
        print(setting, a, f"{len(items)}/{expected[a]} saved", dict(Counter(i["eval"] for i in items)))

    # 2. InjecAgent's own scores.
    # Read only "Valid Rate", "Direct Harm" and "S1". The "S2", "Data Stealing" and "Total" lines assume
    # step two of data stealing was run. We stop after step one, so they count no data-stealing successes.
    print(json.dumps(get_score(files), indent=1))

    # 3. Our combined numbers for this setting (direct harm + first step of data stealing).
    # We divide by all 1,054 cases, so a case the script skipped counts as a failed attack.
    n_total = sum(expected.values())
    n_succ = sum(i["eval"] == "succ" for i in all_items)
    n_valid = sum(i["eval"] in ("succ", "unsucc") for i in all_items)
    print(f"{setting}: valid rate {n_valid / n_total:.1%}, "
          f"ASR-all {n_succ / n_total:.1%}, ASR-valid {n_succ / n_valid:.1%} "
          f"({n_succ} successful, {n_valid} valid, {n_total} cases)\n")

In [ ]:
# Compare the full run with the pilot on the same 60 cases. With no sampling, the responses should match.
pilot_dir = "/content/drive/MyDrive/cs182/pilot"
same = different = missing = 0
for a in ["dh", "ds"]:
    for s in ["base", "enhanced"]:
        full = {(i["User Instruction"], i["Tool Response"]): i
                for i in map(json.loads, open(f"{res_dir}/test_cases_{a}_{s}.json"))}
        for p in map(json.loads, open(f"{pilot_dir}/test_cases_{a}_{s}.json")):
            f = full.get((p["User Instruction"], p["Tool Response"]))
            if f is None:
                missing += 1
            elif f["output"] == p["output"]:
                same += 1
            else:
                different += 1
print(f"identical: {same}, different: {different}, not in full run: {missing}")